# Quick Start

Minimal end-to-end example: define tools, describe the task, plan a workflow, and run it.


## 1. Define tools


In [3]:
from pydantic import BaseModel, Field
from workflow_auto_assembler import create_avc_items, LlmFunctionItemInput

class AddInput(BaseModel):
    x: int = Field(..., description="Number to increment.")

class AddOutput(BaseModel):
    y: int = Field(..., description="Incremented number.")

def add_one(inputs: AddInput) -> AddOutput:
    return AddOutput(y=inputs.x + 1)

class DoubleInput(BaseModel):
    y: int = Field(..., description="Number to double.")

class DoubleOutput(BaseModel):
    z: int = Field(..., description="Doubled number.")

def double(inputs: DoubleInput) -> DoubleOutput:
    return DoubleOutput(z=inputs.y * 2)

available_tools = create_avc_items(functions=[
    LlmFunctionItemInput(func=add_one, input_model=AddInput, output_model=AddOutput),
    LlmFunctionItemInput(func=double, input_model=DoubleInput, output_model=DoubleOutput),
])


## 2. Define task and I/O models


In [4]:
from pydantic import BaseModel, Field

task_description = "Add 1 to the input, then double it."

class WfInputs(BaseModel):
    x: int = Field(..., description="Starting number.")

class WfOutputs(BaseModel):
    result: int = Field(..., description="Final result.")


## 3. Plan workflow


In [5]:
from workflow_auto_assembler import WorkflowAutoAssembler

wa = WorkflowAutoAssembler(
    available_functions=available_tools["available_functions"],
    available_callables=available_tools["available_callables"],
    llm_handler_params={
        "llm_h_type": "ollama",
        "llm_h_params": {
            "connection_string": "http://localhost:11434",
            "model_name": "gpt-oss:20b"
        }
    },
)

wf_obj = await wa.plan_workflow(
    task_description=task_description,
    input_model=WfInputs,
    output_model=WfOutputs,
)
wf_obj.workflow


[{'id': 1,
  'func_id': 'e76db85b541be868911c481cb3ae2749399aff63ae5814bd3e2bb1c0e6837bb8',
  'name': 'add_one',
  'args': {'x': '0.output.x'}},
 {'id': 2,
  'func_id': '60a8f39cc390fc7c067c93f07579ae49b39e794c83db1efce10d5baeab194d14',
  'name': 'double',
  'args': {'y': '1.output.y'}},
 {'id': 3,
  'func_id': '1eeca28b15d68412d08d2b2bc826436955411f021d433c269f4d3dd5b952047c',
  'name': 'output_model',
  'args': {'result': '2.output.z'}}]

## 4. Run workflow


In [6]:
output = await wa.run_workflow(
    workflow_object=wf_obj,
    run_inputs=WfInputs(x=3),
)
output


WfOutputs(result=8)